# Verificação do pré-processamento

**Fase DSML:** II.4–2.6 · **Tarefas:** T1.7–T1.10 · **Dono:** _(a definir)_

Bancada de trabalho: a lógica reutilizável fica em `src/`; aqui corre-se, explora-se e
guarda-se o que o `Projeto_Final.ipynb` vai ler (`reports/experiments.csv`, `reports/figures/`).
Regras da modelação: [docs/modelacao_base.md](../docs/modelacao_base.md).

Só **mostra e valida** o que o `src/` faz; não define regras novas (essas ficam em `docs/decisions.md`).

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # os caminhos de src/ (data/, reports/, models/) são relativos à raiz

import pandas as pd

from src.features.build import CATEGORICAL_FEATURES, DROPPED, FEATURES
from src.pipeline import build_preprocessor, cv_splits, load_model_data

data = load_model_data()

## 2.4 População e split por doente (D1–D7, D13)

In [ ]:
pd.DataFrame({
    "encontros": [len(data.X_train), len(data.X_test)],
    "doentes": [data.groups_train.nunique(), None],
    "positivos": [data.y_train.mean(), data.y_test.mean()],
}, index=["treino", "teste"])

In [ ]:
pd.DataFrame([{"fold": k, "n_val": len(va), "positivos_val": data.y_train.iloc[va].mean(),
               "doentes_partilhados": len(set(data.groups_train.iloc[tr]) & set(data.groups_train.iloc[va]))}
              for k, (tr, va) in enumerate(cv_splits(data))])

## 2.5 Features do modelo (D14, D15)

In [ ]:
print(len(FEATURES), "features;", len(DROPPED), "colunas excluídas")
pd.Series(DROPPED, name="motivo").to_frame()

## 2.6 Pré-processamento (D16)

Ajustado só no treino. Aqui só para inspecionar as colunas geradas.

In [ ]:
pre = build_preprocessor().fit(data.X_train)
names = pd.Series(pre.get_feature_names_out())
print(len(names), "colunas depois do pré-processamento")
names.str.split("_").str[0].value_counts().head(15)